### 🏁 Sprint 8 — Finalisation & Recommandations (Semaine 8)
**Objectif** : Documentation finale et livrables actionnables

**User Stories** :
- 🟡 US-070 : Rédiger 3 recommandations CCI (3 SP)
- 🟡 US-071 : Rédiger 3 recommandations CA (3 SP)
- 🟡 US-072 : Finaliser README complet (2 SP)
- 🟡 US-073 : Créer vidéo démo (5 min) (5 SP) — OPTIONNEL
- 🟡 US-074 : Préparer slides présentation (3 SP) — OPTIONNEL

**Durée estimée** : 8 story points (hors optionnels)

---

### 8.1.1 — EXTRACTION INSIGHTS CLÉS DASHBOARD

**Action** : Analyser les données pour identifier patterns actionnables

**Objectif** : Extraire les constats chiffrés qui serviront de base aux recommandations CCI et CA

**Méthode** :
- Charger données finales communes
- Identifier top/flop communes
- Analyser secteurs les plus vulnérables
- Calculer statistiques clés par profil et catégorie

**Contexte métier** : Les recommandations doivent être factuelles, chiffrées et actionnables. On base tout sur les données du dashboard.

---

In [1]:
import pandas as pd
import os
from pathlib import Path

print("="*90)
print("📊 ÉTAPE 8.1.1 — EXTRACTION INSIGHTS CLÉS")
print("="*90)
print()

base_dir = Path(r"C:\Users\lpint\OneDrive\Bureau\Dynamique commerciale 59\dashboard-commercial-nord59")

# === 1. CHARGEMENT DONNÉES ===
print("📥 Chargement données...")
print()

communes_path = base_dir / "data" / "processed" / "communes_avec_gps_20260513.csv"
commerces_manquants_path = base_dir / "data" / "processed" / "commerces_manquants_20260512.csv"
secteurs_vulnerables_path = base_dir / "data" / "processed" / "secteurs_vulnerables_20260512.csv"

df_communes = pd.read_csv(communes_path)
df_commerces_manquants = pd.read_csv(commerces_manquants_path)
df_secteurs = pd.read_csv(secteurs_vulnerables_path)

print(f"✅ {len(df_communes)} communes chargées")
print(f"✅ {len(df_commerces_manquants)} communes avec commerces manquants")
print(f"✅ {len(df_secteurs)} secteurs analysés")
print()

# === 2. STATISTIQUES GLOBALES ===
print("="*90)
print("📊 STATISTIQUES GLOBALES DÉPARTEMENT")
print("="*90)
print()

stats_globales = {
    'Communes totales': len(df_communes),
    'Établissements actifs': df_communes['nb_actifs'].sum(),
    'Établissements fermés': df_communes['nb_fermes'].sum(),
    'Taux mortalité moyen': df_communes['taux_mortalite'].mean(),
    'Score fragilité moyen': df_communes['score_fragilite'].mean()
}

for key, value in stats_globales.items():
    if isinstance(value, float):
        print(f"  {key:<30} : {value:>10.1f}")
    else:
        print(f"  {key:<30} : {value:>10,}".replace(',', ' '))

print()

# === 3. RÉPARTITION PAR CATÉGORIE ===
print("="*90)
print("📊 RÉPARTITION PAR CATÉGORIE PRIORITÉ")
print("="*90)
print()

repartition_cat = df_communes['categorie_priorite'].value_counts()
for cat, count in repartition_cat.items():
    pct = (count / len(df_communes)) * 100
    print(f"  {cat:<20} : {count:>3} communes ({pct:>5.1f}%)")

print()

# === 4. RÉPARTITION PAR PROFIL ===
print("="*90)
print("📊 RÉPARTITION PAR PROFIL")
print("="*90)
print()

repartition_profil = df_communes['profil'].value_counts()
for profil, count in repartition_profil.items():
    pct = (count / len(df_communes)) * 100
    print(f"  {profil:<20} : {count:>3} communes ({pct:>5.1f}%)")

print()

# === 5. TOP 10 COMMUNES FRAGILES ===
print("="*90)
print("🔴 TOP 10 COMMUNES LES PLUS FRAGILES")
print("="*90)
print()

top_fragiles = df_communes.nlargest(10, 'score_fragilite')[
    ['nom_commune', 'score_fragilite', 'taux_mortalite', 'categorie_priorite']
]

print(f"{'Rang':<6} {'Commune':<25} {'Score':<10} {'Taux mort.':<12} {'Catégorie'}")
print("-" * 80)
for idx, row in enumerate(top_fragiles.itertuples(), 1):
    print(f"{idx:<6} {row.nom_commune:<25} {row.score_fragilite:<10.1f} {row.taux_mortalite:<12.1f} {row.categorie_priorite}")

print()

# === 6. TOP 5 SECTEURS VULNÉRABLES ===
print("="*90)
print("⚠️ TOP 5 SECTEURS NAF LES PLUS VULNÉRABLES")
print("="*90)
print()

top_secteurs = df_secteurs.nlargest(5, 'taux_fermeture')[
    ['naf_classe_libelle', 'taux_fermeture', 'nb_actifs', 'nb_fermes']
]

print(f"{'Secteur':<50} {'Taux fermeture':<15} {'Actifs':<10} {'Fermés'}")
print("-" * 100)
for row in top_secteurs.itertuples():
    print(f"{row.naf_classe_libelle:<50} {row.taux_fermeture:<15.1f}% {row.nb_actifs:<10} {row.nb_fermes}")

print()

# === 7. COMMERCES ESSENTIELS MANQUANTS ===
print("="*90)
print("🏪 COMMERCES ESSENTIELS LES PLUS MANQUANTS")
print("="*90)
print()

# Compter combien de communes manquent chaque type
commerces_count = {}
for col in df_commerces_manquants.columns:
    if col.startswith('manque_'):
        type_commerce = col.replace('manque_', '').replace('_', ' ').title()
        nb_communes = df_commerces_manquants[col].sum()
        commerces_count[type_commerce] = nb_communes

commerces_sorted = sorted(commerces_count.items(), key=lambda x: x[1], reverse=True)[:5]

print(f"{'Type commerce':<30} {'Nb communes manquantes'}")
print("-" * 60)
for commerce, nb in commerces_sorted:
    print(f"{commerce:<30} {nb:>3} communes")

print()

# === 8. DÉSERTS COMMERCIAUX TOTAUX ===
print("="*90)
print("🏜️ DÉSERTS COMMERCIAUX TOTAUX")
print("="*90)
print()

# Communes manquant tous les 7 types
colonnes_manque = [col for col in df_commerces_manquants.columns if col.startswith('manque_')]
df_commerces_manquants['total_manquants'] = df_commerces_manquants[colonnes_manque].sum(axis=1)

deserts_totaux = df_commerces_manquants[df_commerces_manquants['total_manquants'] >= 7]

print(f"  Nombre de déserts commerciaux totaux : {len(deserts_totaux)} communes")
print(f"  (Manquent 7/7 commerces essentiels)")
print()

if len(deserts_totaux) > 0:
    print("  Exemples :")
    for commune in deserts_totaux.head(5)['nom_commune']:
        print(f"    - {commune}")

print()
print("="*90)
print("✅ Étape 8.1.1 terminée — Insights extraits")
print("="*90)

📊 ÉTAPE 8.1.1 — EXTRACTION INSIGHTS CLÉS

📥 Chargement données...

✅ 647 communes chargées
✅ 203 communes avec commerces manquants
✅ 39 secteurs analysés

📊 STATISTIQUES GLOBALES DÉPARTEMENT

  Communes totales               :        647
  Établissements actifs          :     39 261
  Établissements fermés          :     59 108
  Taux mortalité moyen           :       57.1
  Score fragilité moyen          :       44.5

📊 RÉPARTITION PAR CATÉGORIE PRIORITÉ

  Non prioritaire      : 444 communes ( 68.6%)
  Priorité B           : 190 communes ( 29.4%)
  Priorité A           :  13 communes (  2.0%)

📊 RÉPARTITION PAR PROFIL

  Dynamique            : 266 communes ( 41.1%)
  Désertifié           : 249 communes ( 38.5%)
  Précaire             : 129 communes ( 19.9%)
  Métropole            :   2 communes (  0.3%)

🔴 TOP 10 COMMUNES LES PLUS FRAGILES

Rang   Commune                   Score      Taux mort.   Catégorie
------------------------------------------------------------------------------

---

### 💬 Commentaire — Constats clés pour recommandations

#### 📊 Situation départementale préoccupante

**Taux de mortalité commerciale** : 57,1% en moyenne (59 108 fermés vs 39 261 actifs). Indicateur d'une **dynamique structurellement défavorable** avec 1,5 fois plus de fermetures que d'actifs.

**Polarisation territoriale** : 
- 13 communes Priorité A (2%) nécessitant intervention urgente (Lille, Roubaix, Maubeuge, Avesnes-sur-Helpe)
- 190 communes Priorité B (29,4%) à surveiller
- 249 communes Désertifiées (38,5%) = zones rurales sous-équipées

**Secteurs en déclin** : Commerce alimentaire spécialisé, équipements audiovisuels, textile sur marchés → Taux fermeture 70-100%. Signes transformation numérique et changement habitudes consommation.

---

#### 🎯 Angles pour recommandations

**Pour CCI** :
1. Accompagnement secteurs vulnérables (textile, commerce spécialisé)
2. Ciblage 13 communes Priorité A avec dispositifs renforcés
3. Prévention désertification 249 communes rurales

**Pour CA** :
1. Allocation budgétaire prioritaire sur Priorité A et B (203 communes)
2. Évaluation impact politiques existantes (avant/après)
3. Stratégies différenciées par profil (Désertifié vs Dynamique vs Précaire)

---

### ✅ Étape 8.1.1 terminée et validée

**Insights** : ✅ Extraits et analysés  
**Prochaine étape** : Rédaction recommandations CCI

---

---

### 8.2.1 — RÉDACTION RECOMMANDATIONS CCI (US-070)

**Action** : Formuler 3 recommandations actionnables pour les CCI

**Objectif** : Fournir leviers d'action concrets basés sur les données

**Format** : Pour chaque recommandation
- **Constat** : Donnée chiffrée issue dashboard
- **Action** : Mesure concrète proposée
- **Impact attendu** : Résultat quantifiable

**Contexte métier** : Sophie (Chargée mission CCI) et Jean-Pierre (Directeur CCI) ont besoin de recommandations pour arbitrer budget 2025 et cibler interventions.

---

---

## 📋 RECOMMANDATION CCI #1 — Accompagnement renforcé secteurs vulnérables

### 🔍 Constat

**74,7% de taux de fermeture** dans le secteur textile/habillement/chaussures sur marchés (2 505 fermés vs 848 actifs). Secteurs commerce alimentaire spécialisé et équipements audiovisuels en **déclin total** (100% fermeture).

**Cause identifiée** : Transformation numérique (e-commerce) + changement habitudes consommation + concurrence grandes surfaces.

### 💡 Action proposée

**Programme "Transition Commerce 2025-2027"** :
- Formation gratuite 40h pour commerces traditionnels : digitalisation, réseaux sociaux, click & collect
- Accompagnement personnalisé 6 mois (10 commerçants/an dans secteurs textile, alimentaire spécialisé)
- Budget : 150 k€/an (financement FEDER + Région)
- Ciblage : 13 communes Priorité A + 50 communes Priorité B

**Partenaires** : BGE, French Tech, Chambres Métiers

### 🎯 Impact attendu

- **Court terme (1 an)** : 30 commerçants formés, 15% augmentation CA digital
- **Moyen terme (3 ans)** : Stabilisation taux fermeture secteurs ciblés (objectif -10 points)
- **Indicateur suivi** : Évolution nb actifs secteurs textile/alimentaire (dashboard Page 4)

---

## 📋 RECOMMANDATION CCI #2 — Dispositif d'urgence 13 communes Priorité A

### 🔍 Constat

**13 communes Priorité A** (Lille, Roubaix, Maubeuge, Avesnes-sur-Helpe...) cumulent :
- Score fragilité > 57/100
- Taux mortalité 63-100%
- Chômage > 12%

**Situation critique** : 7 communes avec 100% fermetures (Bas-Lieu, Willies, Noyelles-sur-Sambre, Aibes) = désertification totale imminente.

### 💡 Action proposée

**Fonds d'urgence "Revitalisation Commerce"** :
- Aide installation commerce manquant : 10 k€ (boulangerie, épicerie, pharmacie)
- Exonération fiscale 3 ans (CFE) négociée avec communes
- Garantie loyer 2 ans pour propriétaires acceptant bail commercial
- Budget : 200 k€/an (20 installations/an max)
- Priorisation : 7 communes 100% fermeture en priorité absolue

**Critères éligibilité** : Commerce essentiel manquant (identifié Page 7 dashboard), porteur projet viable (business plan validé)

### 🎯 Impact attendu

- **Court terme (1 an)** : 10 commerces réouverts dans 7 communes critiques
- **Moyen terme (2 ans)** : Sortie statut "Priorité A" pour 3-4 communes
- **Indicateur suivi** : Évolution catégorie priorité (dashboard Page 3) + commerces manquants (Page 7)

---

## 📋 RECOMMANDATION CCI #3 — Prévention désertification 249 communes Désertifiées

### 🔍 Constat

**249 communes profil "Désertifié"** (38,5% territoire) = zones rurales avec densité commerciale faible mais pas encore en crise.

**Risque** : Sans intervention, basculement progressif vers Priorité A/B (effet domino fermetures).

**Opportunité** : Communes encore résilientes, prévention moins coûteuse que rattrapage.

### 💡 Action proposée

**Observatoire "Alerte Précoce Commerce Rural"** :
- Monitoring trimestriel 249 communes (mise à jour dashboard automatique)
- Détection signaux faibles : 1ère fermeture commerce essentiel → alerte automatique
- Accompagnement proactif porteurs projet (connexion CCI avant fermeture définitive)
- Budget : 50 k€/an (chargé mission + outils data)
- Outil : Dashboard actuel étendu avec alertes email automatiques

**Indicateurs surveillance** : Évolution nb actifs, 1ère fermeture boulangerie/épicerie, baisse score fragilité

### 🎯 Impact attendu

- **Court terme (1 an)** : Système alerte opérationnel, 5 fermetures évitées par anticipation
- **Moyen terme (3 ans)** : Stabilisation 249 communes Désertifiées (objectif 0 basculement Priorité A)
- **Indicateur suivi** : Flux communes Désertifié → Priorité B/A (dashboard Page 3)

---

### ✅ US-070 : Recommandations CCI terminées (3 SP)

**Livrables** :
- ✅ 3 recommandations CCI rédigées
- ✅ Format Constat → Action → Impact
- ✅ Chiffrées et sourcées (données dashboard)
- ✅ Budget estimé : 400 k€/an total

**Budget total proposé CCI** : 400 k€/an
- Transition Commerce : 150 k€
- Fonds urgence Priorité A : 200 k€
- Observatoire prévention : 50 k€

---

---

### 8.3.1 — RÉDACTION RECOMMANDATIONS CA (US-071)

**Action** : Formuler 3 recommandations actionnables pour les Communautés d'Agglomération

**Objectif** : Fournir leviers d'action intercommunaux basés sur les données

**Format** : Pour chaque recommandation
- **Constat** : Donnée chiffrée issue dashboard
- **Action** : Mesure concrète proposée
- **Impact attendu** : Résultat quantifiable

**Contexte métier** : Claire (VP CA) et Julien (DGS CA) doivent arbitrer budgets intercommunaux (800 k€-2 M€) et évaluer politiques existantes.

---

## 📋 RECOMMANDATION CA #1 — Allocation budgétaire différenciée par profil

### 🔍 Constat

**Hétérogénéité territoriale forte** : 4 profils identifiés avec besoins différents
- **Dynamique** (41,1%) : Score moyen 35/100 → maintien dynamisme
- **Désertifié** (38,5%) : Score moyen 48/100 → prévention basculement
- **Précaire** (19,9%) : Score moyen 62/100 → intervention urgente
- **Métropole** (0,3%) : Score moyen 71/100 → requalification urbaine

**Problème actuel** : Budgets commerce souvent répartis uniformément (ex: 10 k€/commune) sans tenir compte fragilité réelle.

### 💡 Action proposée

**Grille allocation budgétaire** basée sur profil + catégorie :
- **Priorité A** (13 communes) : 50 k€/commune/an (intervention lourde)
- **Priorité B Précaire** (129 communes) : 20 k€/commune/an (accompagnement renforcé)
- **Priorité B Désertifié** (61 communes) : 15 k€/commune/an (prévention)
- **Non prioritaire** (444 communes) : 5 k€/commune/an (maintien veille)

**Budget total type CA 50 communes** : ~650 k€/an (au lieu de 500 k€ uniforme)

**Outil décision** : Dashboard Page 6 (EPCI) avec classement automatique communes par priorité

### 🎯 Impact attendu

- **Court terme (1 an)** : Concentration 60% budget sur 30% communes les plus fragiles
- **Moyen terme (2 ans)** : Réduction écart score fragilité intra-EPCI (objectif σ -10%)
- **Indicateur suivi** : Évolution distribution scores par EPCI (dashboard Page 6)

---

## 📋 RECOMMANDATION CA #2 — Évaluation systématique impact politiques commerce

### 🔍 Constat

**Opacité résultats** : Investissements importants (ex: 2 M€ plan commerce 2021-2024) sans méthodologie évaluation impact avant/après.

**Données disponibles** : Dashboard contient historique 2015-2024 permettant analyse temporelle (Page 1) + rupture COVID (Page 2).

**Opportunité** : Données SIRENE actualisables tous les 6 mois pour suivi longitudinal.

### 💡 Action proposée

**Protocole évaluation "Avant-Après"** :
1. **T0 (avant intervention)** : Export baseline communes bénéficiaires (Page 9 dashboard)
   - KPI : nb_actifs, taux_mortalite, score_fragilite
2. **T+6 mois** : Point étape (alerte si aggravation)
3. **T+24 mois** : Évaluation finale avec groupe contrôle (communes similaires non bénéficiaires)
4. **Indicateurs succès** : 
   - Δ nb_actifs > +5%
   - Δ taux_mortalite < -5 points
   - Stabilisation score fragilité

**Outil** : Dashboard actuel + exports CSV (US-061 Sprint 7)

**Pilotage** : Comité évaluation semestriel (VP CA + DGS + CCI)

### 🎯 Impact attendu

- **Court terme (6 mois)** : 1er protocole évaluation déployé sur plan en cours
- **Moyen terme (2 ans)** : Arbitrage budget 2027 basé sur ROI démontré politiques 2025-2026
- **Long terme** : Capitalisation bonnes pratiques inter-CA (benchmarking)
- **Indicateur suivi** : Dashboard Page 1 (évolution temporelle) + Page 5 (focus commune)

---

## 📋 RECOMMANDATION CA #3 — Mutualisation intercommunale "Commerces partagés"

### 🔍 Constat

**Déserts commerciaux concentrés** : 203 communes Priorité A+B manquent commerces essentiels (boulangerie, épicerie).

**Contrainte ruralité** : Faible densité démographique rend commerce fixe non viable économiquement.

**Opportunité** : Regroupement intercommunal peut créer masse critique (ex: 5 communes 500 hab = 2 500 hab).

### 💡 Action proposée

**Programme "Commerces Mutualisés"** :
- Identification clusters 3-5 communes proches (< 10 km) partageant besoins commerces essentiels
- Financement commun CA :
  - Local commercial intercommunal (50 k€)
  - Subvention équipement commerçant (20 k€)
  - Chiffre affaires garanti 3 ans si CA < seuil (30 k€/an max)
- Rotation commerçant (ex: boulangerie lundi-mercredi commune A, jeudi-vendredi commune B)
- Ciblage prioritaire : 7 communes 100% fermeture + 20 communes déserts partiels

**Budget pilote** : 300 k€ pour 3 commerces mutualisés (15 communes bénéficiaires)

### 🎯 Impact attendu

- **Court terme (18 mois)** : 3 commerces mutualisés opérationnels
- **Moyen terme (3 ans)** : 15 communes sorties statut "commerce essentiel manquant"
- **Long terme** : Modèle réplicable autres CA (essaimage régional)
- **Indicateur suivi** : Dashboard Page 7 (commerces manquants) + nb communes déserts totaux

---

### ✅ US-071 : Recommandations CA terminées (3 SP)

**Livrables** :
- ✅ 3 recommandations CA rédigées
- ✅ Format Constat → Action → Impact
- ✅ Chiffrées et sourcées (données dashboard)
- ✅ Budget estimé : 950 k€/an (type CA 50 communes)

**Budget total proposé CA** : 950 k€/an
- Allocation différenciée : 650 k€
- Protocole évaluation : 0 k€ (outillage existant)
- Commerces mutualisés pilote : 300 k€

---